In [ ]:
import os, requests, pandas as pd
# Import game data
url = "https://api.collegefootballdata.com/games"

api_key = os.environ["COLLEGE_DATA"]

games_df = pd.concat(
    pd.DataFrame(
        requests.get(
            url,
            params={"year": y},
            headers={"Authorization": f"Bearer {api_key}"}
        ).json()
    )
    for y in range(2015, 2027)  # 2015 to 2026 inclusive
)

games_df.to_csv("all_games.csv", index=False)


In [3]:
# import sp+ data

url = "https://api.collegefootballdata.com/ratings/sp"

sp_df = pd.concat(
    pd.DataFrame(
        requests.get(
            url,
            params={"year": y},
            headers={"Authorization": f"Bearer {api_key}"}
        ).json()
    )
    for y in range(2015, 2027)  # 2013 to 2026 inclusive
)

sp_df2 = sp_df[['year','team','rating']]

In [5]:
import numpy as np
# Clean up data
reg_season = games_df.copy()
reg_season = reg_season[reg_season['seasonType'] == 'regular']
reg_season = reg_season[reg_season['season'] != 2026]

# separate home v away so that each team has their own section, to allow for easy aggregation
games_home = reg_season[['season','homeTeam','homeConference','homeClassification','homePoints','awayPoints']]
games_home.columns = ['year','team','conference','classification','scored','allowed']

games_away = reg_season[['season','awayTeam','awayConference','awayClassification','awayPoints','homePoints',]]
games_away.columns = ['year','team','conference','classification','scored','allowed']

# Vertically merge, only keep rows with classification as fbs
games_clean = pd.concat([games_home, games_away])

# Develop total win column if scored > allowed
games_clean['outcome'] = np.where(games_clean['scored'] > games_clean['allowed'], 1, 0)

games_clean = games_clean[games_clean['classification'] == 'fbs'].drop(columns='classification')
# Aggregate total points scored, along with games column
games_clean = games_clean.groupby(['year','team','conference']).agg(
    scored = ('scored','sum'),
    allowed = ('allowed','sum'),
    wins = ('outcome','sum'),
    games=('team','size')
).reset_index()

# Calculate Pythagorean win probability
games_clean['win_rate'] = games_clean['wins'] / games_clean['games']
games_clean['exp_win_rate'] = 1 / (1 + (games_clean['allowed'] / games_clean['scored'])**2.37)
games_clean['exp_win'] = games_clean['exp_win_rate'] * games_clean['games']
# Add SP+ Data
games_clean = pd.merge(games_clean, sp_df2, on=['year','team'])

games_clean.to_csv("game_data.csv", index=False)

In [ ]:
import os, requests, pandas as pd
# Import game data
url = "https://api.collegefootballdata.com/teams"

api_key = os.environ["COLLEGE_DATA"]

teams = pd.DataFrame(
        requests.get(
            url,
            params={"year": 2026},
            headers={"Authorization": f"Bearer {api_key}"}
        ).json()
)

teams = teams[teams['classification'] == 'fbs']
teams.to_csv("team_data.csv", index=False)


In [3]:
import pandas as pd
team = pd.read_csv('team_data.csv')

team[team['conference'] == 'FBS Independents']

,id,school,mascot,abbreviation,alternateNames,conference,division,classification,color,alternateColor,logos,twitter,location
80,87,Notre Dame,Fighting Irish,ND,"['ND', 'Notre Dame']",FBS Independents,NaN,fbs,#0c2340,#c99700,['http://a.espncdn.com/i/teamlogos/ncaa/500/87...,@NDFootball,"{'id': 3855, 'name': 'Notre Dame Stadium', 'ci..."
119,41,UConn,Huskies,CONN,"['Connecticut', 'CONN', 'UConn']",FBS Independents,NaN,fbs,#0c2340,#f1f2f3,['http://a.espncdn.com/i/teamlogos/ncaa/500/41...,@UConnFootball,"{'id': 3892, 'name': 'Pratt & Whitney Stadium'..."
